### Identifying Data Types

Before fixing anything, we need to know what type each column currently 
is. Pandas automatically assigns a type when loading data, but it can 
sometimes get it wrong (e.g. a number stored as text becomes "object").

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("customer_churn.csv")
print(df.dtypes)

CustomerID           object
Age                 float64
Gender               object
TenureYears           int64
MonthlyCharges      float64
TotalCharges        float64
ContractType         object
PaymentMethod        object
InternetService      object
TechSupport          object
SeniorCitizen         int64
Partner              object
PaperlessBilling     object
Churn                object
dtype: object


### Numerical Data

Numerical data represents quantities - either whole numbers (integers) 
or decimal numbers (floats). Example: Age, MonthlyCharges, TotalCharges.

These columns should be stored as `int64` or `float64`, not as text.

In [2]:
numerical_cols = ["Age", "TenureYears", "MonthlyCharges", "TotalCharges"]
print(df[numerical_cols].dtypes)

Age               float64
TenureYears         int64
MonthlyCharges    float64
TotalCharges      float64
dtype: object


### Categorical Data

Categorical data represents a limited set of groups/labels, not 
quantities. Example: ContractType, PaymentMethod, Gender.

These are usually stored as `object` (text) type in pandas.

In [4]:
categorical_cols = ["Gender", "ContractType", "PaymentMethod", "InternetService"]
for col in categorical_cols:
    print(col, "->", df[col].unique())

Gender -> ['male' 'Male' 'M' 'Female']
ContractType -> ['Two Year' 'One Year' 'Month-to-Month']
PaymentMethod -> ['Mailed Check' 'Bank Transfer' 'Credit Card' 'Electronic Check']
InternetService -> ['No' 'Fiber Optic' 'DSL']


### Boolean Data

Boolean data has only two possible values - True/False, or something 
that means the same thing, like Yes/No or 1/0.

Example: TechSupport, Partner, PaperlessBilling, Churn are all Yes/No 
columns, and SeniorCitizen is a 0/1 column - both represent boolean 
information but are stored differently.

In [5]:
print(df["TechSupport"].unique())
print(df["SeniorCitizen"].unique())

# Convert Yes/No column into actual boolean type
df["TechSupport_bool"] = df["TechSupport"].map({"Yes": True, "No": False})
print(df["TechSupport_bool"].dtype)

['Yes' 'No']
[1 0]
bool


### Date/Time Data

Date/Time data represents a specific point in time, like a signup date 
or billing date. Our current dataset doesn't have a date column, so we 
create one here to practice handling this type - this is common in real 
telecom datasets (e.g. "JoinDate").

In [6]:
# Create an example date column to demonstrate date/time handling
np.random.seed(1)
df["JoinDate"] = pd.date_range(start="2015-01-01", periods=len(df), freq="h")
print(df["JoinDate"].dtype)
df[["JoinDate"]].head()

datetime64[ns]


,JoinDate
0,2015-01-01 00:00:00
1,2015-01-01 01:00:00
2,2015-01-01 02:00:00
3,2015-01-01 03:00:00
4,2015-01-01 04:00:00


### String Data

String data is free-form text, not a fixed set of categories. 
Example: CustomerID is a string - it's an identifier, not something we 
analyze the "value" of.

In [7]:
print(df["CustomerID"].dtype)
print(df["CustomerID"].head())

object
0    CUST00001
1    CUST00002
2    CUST00003
3    CUST00004
4    CUST00005
Name: CustomerID, dtype: object


### Converting Data Types - astype()

`astype()` is used to directly convert a column from one type to another. 
It works well when the data is already clean, but it will throw an error 
if it finds a value it cannot convert (e.g. text mixed with numbers).

In [8]:
# Example: convert SeniorCitizen from int to a proper category type
df["SeniorCitizen"] = df["SeniorCitizen"].astype("category")
print(df["SeniorCitizen"].dtype)

category


### Converting Data Types - to_numeric()

`pd.to_numeric()` converts a column to a numeric type. Unlike `astype()`, 
it can handle invalid values gracefully using `errors="coerce"`, which 
turns bad values into NaN instead of crashing.

In [9]:
# TotalCharges has some values stored as text with extra spaces (from our data creation step)
print(df["TotalCharges"].dtype)

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print(df["TotalCharges"].dtype)
print("Values that failed conversion (now NaN):", df["TotalCharges"].isnull().sum())

float64
float64
Values that failed conversion (now NaN): 151


### Converting Data Types - to_datetime()

`pd.to_datetime()` converts a column into proper date/time format, so we 
can do date-based operations like extracting year, month, or calculating 
duration between two dates.

In [10]:
df["JoinDate"] = pd.to_datetime(df["JoinDate"], errors="coerce")
print(df["JoinDate"].dtype)

# Extract useful parts from the date
df["JoinYear"] = df["JoinDate"].dt.year
df["JoinMonth"] = df["JoinDate"].dt.month
df[["JoinDate", "JoinYear", "JoinMonth"]].head()

datetime64[ns]


,JoinDate,JoinYear,JoinMonth
0,2015-01-01 00:00:00,2015,1
1,2015-01-01 01:00:00,2015,1
2,2015-01-01 02:00:00,2015,1
3,2015-01-01 03:00:00,2015,1
4,2015-01-01 04:00:00,2015,1


### Handling Invalid Conversions

When converting types, some values may not fit the target type 
(e.g. text like "N/A" inside a numeric column). Using `errors="coerce"` 
converts these bad values to NaN instead of stopping the whole process 
with an error. We then decide how to handle those NaNs later (Notebook 3).

In [11]:
# Example: simulate an invalid value and see how coerce handles it
sample = pd.Series(["100", "200", "abc", "300"])
converted = pd.to_numeric(sample, errors="coerce")
print(converted)

0    100.0
1    200.0
2      NaN
3    300.0
dtype: float64


### Detecting Incorrect Data Types

A column might look numeric but actually be stored as text (object) - 
this is a common hidden problem. We detect it by checking `dtypes` and 
trying to spot columns that should be numeric/date but are shown as 
`object`.

In [12]:
# Detect object columns that might actually be numeric
for col in df.select_dtypes(include="object").columns:
    sample_values = df[col].dropna().unique()[:5]
    print(col, "->", sample_values)

CustomerID -> ['CUST00001' 'CUST00002' 'CUST00003' 'CUST00004' 'CUST00005']
Gender -> ['male' 'Male' 'M' 'Female']
ContractType -> ['Two Year' 'One Year' 'Month-to-Month']
PaymentMethod -> ['Mailed Check' 'Bank Transfer' 'Credit Card' 'Electronic Check']
InternetService -> ['No' 'Fiber Optic' 'DSL']
TechSupport -> ['Yes' 'No']
Partner -> ['Yes' 'No']
PaperlessBilling -> ['Yes' 'No']
Churn -> ['Yes' 'No']


### Example: String -> Numeric

This shows converting a text-based numeric column into a real numeric 
column, which we already did for TotalCharges above.

In [13]:
print("TotalCharges dtype now:", df["TotalCharges"].dtype)
df["TotalCharges"].describe()

TotalCharges dtype now: float64


count    4909.000000
mean     4538.654661
std      2591.905353
min        22.270000
25%      2296.880000
50%      4564.540000
75%      6799.240000
max      8999.320000
Name: TotalCharges, dtype: float64

### Example: String -> Date

This shows converting a text-based date column into a real datetime 
column, which we already did for JoinDate above.

In [14]:
print("JoinDate dtype now:", df["JoinDate"].dtype)
df["JoinDate"].describe()

JoinDate dtype now: datetime64[ns]


count                   5060
mean     2015-04-16 09:30:00
min      2015-01-01 00:00:00
25%      2015-02-22 16:45:00
50%      2015-04-16 09:30:00
75%      2015-06-08 02:15:00
max      2015-07-30 19:00:00
Name: JoinDate, dtype: object

### Example: Numeric -> Categorical

SeniorCitizen is stored as 0/1, but it actually represents a category 
(Senior or Not Senior), not a quantity that can be added or averaged. 
Converting it to category type makes this clearer.

In [15]:
df["SeniorCitizen_label"] = df["SeniorCitizen"].map({0: "Not Senior", 1: "Senior"}).astype("category")
print(df["SeniorCitizen_label"].dtype)
df["SeniorCitizen_label"].value_counts()

category


SeniorCitizen_label
Senior        2622
Not Senior    2438
Name: count, dtype: int64

### Example: Integer -> Float

Sometimes a numeric column needs to be a float instead of an integer, 
especially when it will later contain decimal values (e.g. after 
imputation with a mean value in Notebook 3).

In [16]:
print("TenureYears dtype before:", df["TenureYears"].dtype)

df["TenureYears"] = df["TenureYears"].astype(float)
print("TenureYears dtype after:", df["TenureYears"].dtype)

TenureYears dtype before: int64
TenureYears dtype after: float64


### Why Correct Data Types Are Important

- Wrong types can cause errors when doing math or statistics on a column.
- Categorical columns stored as numbers might be wrongly treated as 
  quantities by a model (e.g. averaging SeniorCitizen 0/1 makes no sense).
- Date columns stored as text can't be used to calculate durations or 
  trends.
- Correct types make later steps (scaling, encoding, aggregating) work 
  properly and give accurate results.

In [17]:
# Final check - confirm all key columns now have correct types
print(df.dtypes)

CustomerID                     object
Age                           float64
Gender                         object
TenureYears                   float64
MonthlyCharges                float64
TotalCharges                  float64
ContractType                   object
PaymentMethod                  object
InternetService                object
TechSupport                    object
SeniorCitizen                category
Partner                        object
PaperlessBilling               object
Churn                          object
TechSupport_bool                 bool
JoinDate               datetime64[ns]
JoinYear                        int32
JoinMonth                       int32
SeniorCitizen_label          category
dtype: object
